# ALM frictional mortar contact condition

This notebook generates `custom_conditions/ALM_frictional_mortar_contact_condition.cpp`, the local left- and
right-hand sides of `AugmentedLagrangianMethodFrictionalMortarContactCondition<TDim, TNumNodes, TNormalVariation, TNumNodesMaster>`,
the **augmented Lagrangian (ALM) frictional** mortar contact condition with a vector Lagrange multiplier
(`VECTOR_LAGRANGE_MULTIPLIER`). Theory: thesis §4.3.4 (V. Mataix Ferrándiz, *Numerical methods for contact problems*, UPC 2020) and the
[Frictional contact](https://kratosmultiphysics.github.io/Kratos/pages/Applications/Contact_Structural_Mechanics_Application/Theory/Frictional_Contact.html)
page of the documentation, whose equation numbers are quoted below.

## Formulation

### Coulomb's law and the augmented multipliers

The Coulomb law relates the tangential traction $\mathbf{t}^\tau_{co}$ to the tangential relative velocity
$\mathbf{v}_{\tau,rel}$ through the non-smooth graph (thesis eq. 4.45)

$$\phi_{co} := \Vert \mathbf{t}^\tau_{co} \Vert - \mu \Vert p_n \Vert \le 0, \qquad \mathbf{v}_{\tau,rel} + \beta\, \mathbf{t}^\tau_{co} = \mathbf{0}, \qquad \beta \ge 0, \qquad \phi_{co}\,\beta = 0$$

(stick when $\phi_{co} < 0$, slip on the Coulomb limit). With the tangential projector and the tangential
multiplier (thesis eq. 4.46)

$$\boldsymbol{\tau} = \mathbf{I} - \mathbf{n} \otimes \mathbf{n}, \qquad \boldsymbol{\lambda}_\tau = \boldsymbol{\lambda} - \mathbf{n}\,\lambda_n, \qquad \lambda_n = \mathbf{n} \cdot \boldsymbol{\lambda}$$

the augmented Lagrangian of Alart and Curnier uses the **augmented normal and tangential multipliers**
(thesis eq. 4.60)

$$\bar{\lambda}_n = k\,\lambda_n + \varepsilon_n\, g_n, \qquad \bar{\boldsymbol{\lambda}}_\tau = k\,\boldsymbol{\lambda}_\tau + \varepsilon_\tau\, \mathbf{v}_{\tau,rel}$$

where $k$ is the scale factor (`ScaleFactor`, `SCALE_FACTOR`), $\varepsilon_n$ the normal penalty
(`PenaltyParameter`, nodal `INITIAL_PENALTY`) and $\varepsilon_\tau = \kappa\,\varepsilon_n$ the tangential
one (`TangentFactor` $= \kappa$, `TANGENT_FACTOR`).

### Weak form

The variation of the augmented Lagrangian has three branches (thesis eq. 4.62):

$$\delta \mathcal{L}_{co} = \int_{\Gamma^1_c} \begin{cases}
\bar{\lambda}_n\, \delta g_n + k\, g_n\, \delta \lambda_n + \bar{\boldsymbol{\lambda}}_\tau \cdot \delta \mathbf{v}_{\tau,rel} + \mathbf{v}_{\tau,rel} \cdot \delta \bar{\boldsymbol{\lambda}}_\tau & \Vert \bar{\boldsymbol{\lambda}}_\tau \Vert \le -\mu \bar{\lambda}_n \quad \text{(contact, stick)} \\[4pt]
\bar{\lambda}_n\, \delta g_n + k\, g_n\, \delta \lambda_n - \mu \bar{\lambda}_n \dfrac{\boldsymbol{\lambda}_\tau}{\Vert \boldsymbol{\lambda}_\tau \Vert} \cdot \delta \mathbf{v}_{\tau,rel} - \dfrac{k \boldsymbol{\lambda}_\tau + \mu \bar{\lambda}_n \frac{\bar{\boldsymbol{\lambda}}_\tau}{\Vert \bar{\boldsymbol{\lambda}}_\tau \Vert}}{\varepsilon_\tau} \cdot \delta \boldsymbol{\lambda}_\tau & \Vert \bar{\boldsymbol{\lambda}}_\tau \Vert > -\mu \bar{\lambda}_n \quad \text{(contact, slip)} \\[4pt]
-\dfrac{k^2}{\varepsilon_n} \lambda_n\, \delta \lambda_n - \dfrac{k^2}{\varepsilon_\tau} \boldsymbol{\lambda}_\tau \cdot \delta \boldsymbol{\lambda}_\tau & \bar{\lambda}_n > 0 \quad \text{(gap)}
\end{cases} \mathrm{d}\Gamma$$

### Discrete slip: objective and non-objective measures

The tangential relative velocity is discretised with the mortar operators (thesis eqs. 4.63-4.64). The naive
projection $\boldsymbol{\tau}_j [\mathbf{D}_j \dot{\mathbf{x}}^1_j - \sum_l \mathbf{M}_l \dot{\mathbf{x}}^2_l]$ (eq. 4.65a) is not frame
indifferent; objectivity is restored by carrying the velocity through the **rates of the mortar operators**
(eq. 4.67b) and, with a backward Euler approximation of those rates (eq. 4.68), the **objective nodal slip
increment** is (thesis eq. 4.69b)

$$\tilde{\mathbf{u}}^{obj}_{\tau,j} = \boldsymbol{\tau}_j \left[ \left( \mathbf{D}^{t+\Delta t} - \mathbf{D}^{t} \right) \mathbf{x}^{1} - \left( \mathbf{M}^{t+\Delta t} - \mathbf{M}^{t} \right) \mathbf{x}^{2} \right]_j$$

which requires the mortar operators of the previous converged step (`DOperatorold`, `MOperatorold`,
`mPreviousMortarOperators`). When the operators do not change between two steps (a matching interface that
only translates tangentially, or a step where the pairing did not change) this increment vanishes
identically, so the code also keeps the displacement-based, **non-objective** increment obtained from
eq. 4.65a,

$$\tilde{\mathbf{u}}^{nonobj}_{\tau,j} = -\,\boldsymbol{\tau}_j \left[ \mathbf{D} \left( \mathbf{x}^{1} - \mathbf{x}^{1,t} \right) - \mathbf{M} \left( \mathbf{x}^{2} - \mathbf{x}^{2,t} \right) \right]_j$$

and switches between them at run time: the objective slip is used when the Frobenius norms of
$\mathbf{D} - \mathbf{D}^t$ and $\mathbf{M} - \mathbf{M}^t$ both exceed `OPERATOR_THRESHOLD` (`is_objetive` in the
generated code, the condition being flagged `MODIFIED` otherwise). The same test drives the explicit
computation of the nodal `WEIGHTED_SLIP` (`MortarExplicitContributionUtilities`), so the residual and the
active-set decisions see the same slip.

**Signs.** Both measures approximate the same tangential relative motion with the convention of the weighted
gap (positive when open): for a slave sliding by $\delta$ over a fixed master both give $-D_{jj}\,\delta$
(the C++ tests `WeightedGap3`/`WeightedGap4` check exactly this). The previous coordinates are
$\mathbf{x}^{1,t} = \mathbf{X}^1 + \mathbf{u}^1_{old}$ where `u1old` is the offset from `X1` (the coordinates
held by `DerivativeData`, which are the previous-step ones after the first step) to the previous position,
so that $\mathbf{x}^1 - \mathbf{x}^{1,t}$ is always the displacement increment of the step, consistently with
`WEIGHTED_SLIP`. The $\Delta t$ of eqs. 4.68-4.69 cancels between the velocity and the increment and is not
needed by the generated code.

The **tangent direction** $\boldsymbol{\tau}_j$ used by the Coulomb traction (`TangentSlave`, nodal `TANGENT_XI`)
is computed by `MortarUtilities::ComputeNodesTangentModelPart`: the direction of the tangential multiplier
for stick nodes and the direction of `WEIGHTED_SLIP` for slip nodes, which makes the friction traction
$-\mu\,\bar{\lambda}_n\,\boldsymbol{\tau}_j$ (with $\bar{\lambda}_n < 0$ in compression) oppose the slave's relative motion.

### Discrete residuals (thesis eq. 4.72b)

For a slave node $j$ the multiplier residuals of the three states are

$$\begin{cases}
\mathbf{r}_{\lambda_{\mathcal{A}_{sl}}} = k\, \mathbf{n} \left( -\mathbf{n} \cdot ( \mathbf{D} \mathbf{x}_1 - \mathbf{M} \mathbf{x}_2 ) \right) - \dfrac{k^2}{\varepsilon_n} \left( \boldsymbol{\tau} \cdot \boldsymbol{\lambda} - \dfrac{\mathscr{F}}{k} \right), \qquad \mathscr{F} = -\mu\, \bar{\lambda}_n\, \boldsymbol{\tau} \\[4pt]
\mathbf{r}_{\lambda_{\mathcal{A}_{st}}} = k\, \mathbf{n} \left( -\mathbf{n} \cdot ( \mathbf{D} \mathbf{x}_1 - \mathbf{M} \mathbf{x}_2 ) \right) + k\, \tilde{\mathbf{u}}_\tau \\[4pt]
\mathbf{r}_{\lambda_\mathcal{I}} = \dfrac{k^2}{\varepsilon_n} \mathbf{n} \cdot \boldsymbol{\lambda} + \dfrac{k^2}{\varepsilon_\tau} \boldsymbol{\tau} \cdot \boldsymbol{\lambda}
\end{cases}$$

and the displacement rows receive the virtual work of the augmented traction $\bar{\boldsymbol{\lambda}}_j$
through $\mathbf{D}\,\delta\mathbf{u}^1 - \mathbf{M}\,\delta\mathbf{u}^2$ (the columns $k\mathbf{D}^T$, $-k\mathbf{M}^T$ of eq. 4.72a).

### The five generated branches

Per slave node $j$ the generated functional $\mathcal{R}_j$ is (with $\mathcal{D}_j$ the `DynamicFactor`,
$\tilde{g}_{n,j}$ the weighted gap, $\delta\boldsymbol{\lambda}_{\tau,j}$ the tangential part of the test multiplier):

| branch | $\mathcal{R}_j$ |
|---|---|
| inactive | $-\dfrac{k^2}{\varepsilon_j} \lambda_{n,j}\, \delta\lambda_{n,j} - \dfrac{k^2}{\kappa\,\varepsilon_j} \boldsymbol{\lambda}_{\tau,j} \cdot \delta\boldsymbol{\lambda}_{\tau,j}$ |
| active slip (objective / non-objective) | $k\, \tilde{g}_{n,j}\, \delta\lambda_{n,j} + \mathcal{D}_j\, \bar{\boldsymbol{\lambda}}_j \cdot ( \mathbf{D}\,\mathbf{w}^1 - \mathbf{M}\,\mathbf{w}^2 )_j - \dfrac{k^2}{\varepsilon_j}\, \delta\boldsymbol{\lambda}_{\tau,j} \cdot \left( \boldsymbol{\lambda}_{\tau,j} - \dfrac{\bar{\mathbf{p}}_{\tau,j}}{k} \right)$, with $\bar{\boldsymbol{\lambda}}_j = k\boldsymbol{\lambda}_j + \varepsilon_j \tilde{g}_{n,j} \mathbf{n}_j$ and $\bar{\mathbf{p}}_{\tau,j} = -\mu_j \bar{\lambda}_{n,j} \boldsymbol{\tau}_j$ |
| active stick (objective / non-objective) | $k\, \tilde{g}_{n,j}\, \delta\lambda_{n,j} + \mathcal{D}_j\, \bar{\boldsymbol{\lambda}}_j \cdot ( \mathbf{D}\,\mathbf{w}^1 - \mathbf{M}\,\mathbf{w}^2 )_j + k\, \tilde{\mathbf{u}}_{\tau,j} \cdot \delta\boldsymbol{\lambda}_{\tau,j}$, with $\bar{\boldsymbol{\lambda}}_j = k\boldsymbol{\lambda}_j + \varepsilon_j \tilde{g}_{n,j} \mathbf{n}_j + \kappa\,\varepsilon_j\, \tilde{\mathbf{u}}_{\tau,j}$ |

Only the stick branches differ between the objective and the non-objective slip in the functional itself,
but since the LHS is the consistent linearisation (the mortar operators are DoF-dependent) both slip variants
are emitted for the slip branches as well. The run-time dispatch of the generated code is, per node,
`if (r_geometry[i].IsNot(ACTIVE)) {...} else if (r_geometry[i].Is(SLIP)) { if (is_objetive) {...} else {...} } else { if (is_objetive) {...} else {...} }`.

## From the functional to the generated C++

All the mechanics of the generation live in `../mortar_condition_generator.py` (see also the
[Automatic differentiation](https://kratosmultiphysics.github.io/Kratos/pages/Applications/Contact_Structural_Mechanics_Application/Theory/Automatic_Differentiation.html)
page, thesis Appendix C):

1. **Symbols** (`SymbolSet`): the nodal unknowns `u1`, `u2` (displacements of the slave and master nodes),
   the multipliers, the test functions `w1`, `w2`, `wLM`, the reference coordinates `X1`, `X2`, the nodal
   normals `NormalSlave`, the mortar operators `DOperator`, `MOperator` and the parameters. The current
   coordinates are $\mathbf{x}^{(i)} = \mathbf{X}^{(i)} + \mathbf{u}^{(i)}$ and the nodal **weighted gap**
   (thesis eq. 4.31) is
   $$\tilde{g}_{n,j} = -\,\mathbf{n}_j \cdot \left( \mathbf{D}\, \mathbf{x}^{(1)} - \mathbf{M}\, \mathbf{x}^{(2)} \right)_j$$
   which is **positive for an open gap** and negative for penetration (`NormalGap` / `WEIGHTED_GAP`).
2. **AD exceptions** (thesis §C.3.1): $\mathbf{D}$, $\mathbf{M}$ and, when `TNormalVariation` is `true`, $\mathbf{n}$
   are not expressed in terms of the displacements. They are declared *undefined functions of the DoFs*
   (`DefineDofDependencyMatrix`), so that the chain rule produces unevaluated derivatives that are mapped to
   the arrays computed at run time by `DerivativesUtilities`:

   | symbolic node | C++ |
   |---|---|
   | `DOperator_i_j(u...)` | `DOperator(i,j)` |
   | `Derivative(DOperator_i_j(u...), u_k)` | `DeltaDOperator[k](i,j)` |
   | `Derivative(MOperator_i_j(u...), u_k)` | `DeltaMOperator[k](i,j)` |
   | `Derivative(NormalSlave_i_j(u...), u_k)` | `DeltaNormalSlave[k](i,j)` (normal variation only) |

   The index `k` runs over the slave displacement DoFs first and then the master ones, the ordering used by
   `MortarOperatorWithDerivatives`.
3. **Differentiation**: for every slave node $j$ and every active-set branch the functional $\mathcal{R}_j$
   returned by the function below is differentiated: $\mathbf{r} = \partial \mathcal{R} / \partial \mathbf{w}$
   (local RHS) and $\mathbf{K} = -\partial \mathbf{r} / \partial \mathbf{d}$ (local LHS), with the DoF vector
   $\mathbf{d} = [\mathbf{u}^{(2)}, \mathbf{u}^{(1)}, \boldsymbol{\lambda}]$ ordered *master, slave, multiplier* exactly as
   `GetDofList`. This is the Kratos convention $\mathbf{K}\,\Delta\mathbf{d} = \mathbf{r}$.
4. **Printing**: the derivative nodes are replaced by plain symbols, `sympy.cse` collects the common factors
   (`clhs*`, `crhs*`) and `sympy.ccode` prints C++; only the non-zero entries are emitted, accumulated with `+=`.
5. **Assembly of the file**: one `CalculateLocalLHS` specialisation per geometry pair (`2D2N`, `3D3N`, `3D4N`,
   `3D3N4N`, `3D4N3N`) and per `TNormalVariation` value; the RHS does **not** depend on the derivatives of the
   normal, so `StaticCalculateLocalRHS` is generated only for `TNormalVariation = false` and the `true`
   specialisation forwards to it. The bodies are substituted into the `*_template.cpp` of this folder at the
   `// replace_lhs` / `// replace_rhs` markers and the result is written once.

**Sign convention of the test-function quantities.** Every `<quantity>w` symbol (`NormalwGap`, `TangentwSlip*`)
is *minus* the variation of the quantity in the direction of the test functions, $X_w = -\delta X$: with the
gap defined as above, `NormalwGap = +n.(D w1 - M w2)`, so that the virtual work of a traction $\mathbf{t}$
is written $\mathbf{t} \cdot X_w$ and the residual is $-\delta\Pi$ (the force acting on the bodies).

## How to run this notebook

* **Requirements**: Python 3 and `sympy` (any modern version, tested with 1.14). A compiled Kratos is *not*
  needed: the shared module `../mortar_condition_generator.py` imports `custom_sympy_fe_utilities.py` and the
  core `sympy_fe_utilities.py` directly from the source tree.
* **Interactively**: open it with Jupyter from this folder and run all cells.
* **Headless** (no Jupyter installed): `python3 ../run_notebook.py <this notebook>` executes the code cells
  with the standard library only.
* The equivalent command-line script `generate_*.py` in this folder contains the *same* functional and
  generation call; keep both in sync when the formulation changes.

The output is written directly into `custom_conditions/` (overwriting the committed file). The generation of
the five geometries and the two normal-variation flags takes from a few minutes (frictionless) to about an
hour (ALM frictional). Restrict `COMBINATIONS` / `NORMAL_VARIATIONS` in the configuration cell for a quick
test, or set `OUTPUT_DIR` to a scratch folder.

In [ ]:
import os
import sys

# The shared generator module lives one folder up (automatic_differentiation/)
sys.path.insert(0, os.path.abspath(".."))
import mortar_condition_generator as generator

print("sympy", generator.sympy.__version__)

## Symbols of `SymbolSet` used by the functional

| symbol | meaning | C++ counterpart |
|---|---|---|
| `s.LM`, `s.LMNormal[j]`, `s.LMTangent.row(j)` | $\boldsymbol{\lambda}_j$, $\lambda_{n,j}$, $\boldsymbol{\lambda}_{\tau,j}$ | `VECTOR_LAGRANGE_MULTIPLIER` |
| `s.wLM`, `s.wLMNormal[j]`, `s.wLMTangent.row(j)` | test multiplier and its components | differentiated away |
| `s.NormalGap[j]` | $\tilde{g}_{n,j}$ | `WEIGHTED_GAP` |
| `s.NormalSlave.row(j)`, `s.TangentSlave.row(j)` | $\mathbf{n}_j$, $\boldsymbol{\tau}_j$ | `NORMAL`, `TANGENT_XI` |
| `s.Dw1Mw2.row(j)` | $(\mathbf{D}\,\mathbf{w}^1 - \mathbf{M}\,\mathbf{w}^2)_j$ | — |
| `s.TangentSlipObjective.row(j)`, `s.TangentSlipNonObjective.row(j)` | $\tilde{\mathbf{u}}^{obj}_{\tau,j}$, $\tilde{\mathbf{u}}^{nonobj}_{\tau,j}$ | `WEIGHTED_SLIP` (nodal, assembled) |
| `s.TangentwSlipObjective`, `s.TangentwSlipNonObjective` | $-\delta\tilde{\mathbf{u}}_\tau$ (convention $X_w = -\delta X$), reserved for the friction virtual-work form | — |
| `s.ScaleFactor`, `s.PenaltyParameter[j]`, `s.TangentFactor`, `s.mu[j]`, `s.DynamicFactor[j]` | $k$, $\varepsilon_j$, $\kappa$, $\mu_j$, $\mathcal{D}_j$ | `SCALE_FACTOR`, `INITIAL_PENALTY`, `TANGENT_FACTOR`, `FRICTION_COEFFICIENT`, `DYNAMIC_FACTOR` |

The branch identifiers passed to the functional are `inactive`, `slip_objective`, `slip_non_objective`,
`stick_objective` and `stick_non_objective`.

## The functional

This is the physics of the condition; it is the only family-specific input of the generator.

In [ ]:
def frictional_functional(s, node, branch):
    """Galerkin functional of one slave node in one active-set branch (thesis eqs. 4.60-4.62, 4.72).

    ``branch`` is one of ``inactive``, ``slip_objective``, ``slip_non_objective``, ``stick_objective``,
    ``stick_non_objective``. The multiplier, penalty and slip quantities are those of ``SymbolSet``.
    """
    rv_galerkin = 0
    if branch == "inactive":
        # The multiplier is penalised to zero (gap zone): normal and tangential components
        rv_galerkin -= s.ScaleFactor**2 / s.PenaltyParameter[node] * s.LMNormal[node] * s.wLMNormal[node]
        rv_galerkin -= s.ScaleFactor**2 / (s.PenaltyParameter[node] * s.TangentFactor) * (s.LMTangent.row(node)).dot(s.wLMTangent.row(node))
    else:
        # Normal contact constraint weighted with the normal test multiplier
        rv_galerkin += s.ScaleFactor * s.NormalGap[node] * s.wLMNormal[node]

        augmented_normal_contact_pressure = (s.ScaleFactor * s.LMNormal[node] + s.PenaltyParameter[node] * s.NormalGap[node])
        augmented_lm = (s.ScaleFactor * s.LM.row(node) + s.PenaltyParameter[node] * s.NormalGap[node] * s.NormalSlave.row(node))

        if branch in ("slip_objective", "slip_non_objective"):
            # Coulomb limit: the tangential multiplier is driven to -mu * p_n * tau
            augmented_tangent_contact_pressure = - s.mu[node] * augmented_normal_contact_pressure * s.TangentSlave.row(node)
            rv_galerkin -= s.ScaleFactor**2 / s.PenaltyParameter[node] * (s.wLMTangent.row(node).dot(s.LMTangent.row(node) - augmented_tangent_contact_pressure / s.ScaleFactor))
        else:
            # Stick: the tangential slip is constrained to zero and augments the multiplier
            tangent_slip = s.TangentSlipObjective if branch == "stick_objective" else s.TangentSlipNonObjective
            augmented_lm += s.TangentFactor * s.PenaltyParameter[node] * tangent_slip.row(node)
            rv_galerkin += s.ScaleFactor * (tangent_slip.row(node)).dot(s.wLMTangent.row(node))

        # Virtual work of the augmented contact traction
        rv_galerkin += s.DynamicFactor[node] * (augmented_lm).dot(s.Dw1Mw2.row(node))

    return rv_galerkin

## Configuration

`COMBINATIONS` holds the `(dimension, slave nodes, master nodes)` triplets to generate and
`NORMAL_VARIATIONS` the values of `TNormalVariation`. The output goes to `custom_conditions/` by default.

In [ ]:
COMBINATIONS = generator.DEFAULT_COMBINATIONS   # ((2, 2, 2), (3, 3, 3), (3, 4, 4), (3, 3, 4), (3, 4, 3))
NORMAL_VARIATIONS = (False, True)
TEMPLATE_DIR, OUTPUT_DIR = generator.DefaultDirectories(os.getcwd())
print("template folder:", TEMPLATE_DIR)
print("output folder  :", OUTPUT_DIR)

## Generation

Every branch reports the size of the local system and the number of non-zero entries. The generated
code is checked for symbolic leftovers before the file is written.

In [ ]:
output_path = generator.Generate(generator.ALM_FRICTIONAL, frictional_functional, TEMPLATE_DIR, OUTPUT_DIR, COMBINATIONS, NORMAL_VARIATIONS)

## Check of the generated file

The file must contain, for each of the geometries and normal-variation flags requested, one
`CalculateLocalLHS` specialisation and one `StaticCalculateLocalRHS` specialisation (a full body for
`false`, a forwarder for `true`).

In [ ]:
import re

with open(output_path) as generated_file:
    generated = generated_file.read()

lhs_specialisations = re.findall(r"^void AugmentedLagrangianMethodFrictionalMortarContactCondition<(\d+),\s*(\d+), (true|false), (\d+)>::CalculateLocalLHS\(", generated, re.MULTILINE)
rhs_specialisations = re.findall(r"^void AugmentedLagrangianMethodFrictionalMortarContactCondition<(\d+),\s*(\d+), (true|false), (\d+)>::StaticCalculateLocalRHS\(", generated, re.MULTILINE)
expected = len(COMBINATIONS) * len(NORMAL_VARIATIONS)
print("{} lines, {} LHS and {} RHS specialisations (expected {} each)".format(generated.count("\n"), len(lhs_specialisations), len(rhs_specialisations), expected))
assert len(lhs_specialisations) == expected and len(rhs_specialisations) == expected
assert "Derivative(" not in generated and "//subsvar_" not in generated